# 의료 단락 임베딩 (BGE-M3, 1024차원)
1. 메뉴 **런타임 → 런타임 유형 변경 → GPU**(A100/L4 등) 선택
2. 위에서부터 셀을 하나씩 실행 (Shift+Enter)
3. 결과 파일 `medical_embeddings.csv`는 구글 드라이브 `MyDrive`에 저장돼요

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U sentence-transformers

## 1. 단락 파일 올리기
실행하면 **파일 선택** 버튼이 나와요 → `medical_passages.parquet` 선택

In [ ]:
from google.colab import files
import pandas as pd

files.upload()
df = pd.read_parquet('medical_passages.parquet')
print(len(df), '개 단락')
df.head()

## 2. 임베딩 만들기
제목 + 본문을 함께 넣어요. GPU면 몇 분이면 끝나요.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-m3', device='cuda')
model.half()                 # GPU 메모리 절약 + 속도
model.max_seq_length = 1024  # 단락이 짧아서 충분함 (최대 약 3,200자)

texts = (df['title'] + '\n' + df['text']).tolist()
emb = model.encode(texts, batch_size=64, normalize_embeddings=True,
                   show_progress_bar=True, convert_to_numpy=True)
print(emb.shape)  # (16368, 1024) 이어야 함

## 3. 구글 드라이브에 저장
처음 실행하면 드라이브 접근 허용 창이 떠요.

In [ ]:
from google.colab import drive
import csv
drive.mount('/content/drive')

out = '/content/drive/MyDrive/medical_embeddings.csv'
with open(out, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    for docid, v in zip(df['docid'], emb.astype('float32')):
        w.writerow([docid, 'BAAI/bge-m3', '[' + ','.join(f'{x:.6f}' for x in v) + ']'])
print('저장 완료:', out)

## 4. (선택) 검색 잘 되는지 맛보기

In [ ]:
q = '당뇨병의 증상은 무엇인가요?'
qv = model.encode([q], normalize_embeddings=True)[0]
top = (emb @ qv).argsort()[::-1][:5]
for i in top:
    print(round(float(emb[i] @ qv), 3), df['docid'][i], df['title'][i], '|', df['text'][i][:80])